<a href="https://colab.research.google.com/github/RichmondAlake/cookbook/blob/main/nb/Qwen_3_5_27B_A100(80GB).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To run this, press "*Runtime*" and press "*Run all*" on your A100 Google Colab Pro instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Introducing **[Unsloth Desktop](https://unsloth.ai/docs/desktop)**, the first desktop app to run and train models. Free and open-source for macOS, Windows and Linux. [GitHub](https://github.com/unslothai/unsloth) • [Download](https://unsloth.ai/download)

<p>
<a href="https://unsloth.ai/docs/desktop"><img src="https://raw.githubusercontent.com/unslothai/notebooks/refs/heads/main/assets/unsloth-qwen3-8.png" width="350" alt="Introducing Unsloth Desktop"></a>
</p>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [1]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.3.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

In [2]:
import os, zipfile
if os.path.exists("calderwood_data.zip"):
    zipfile.ZipFile("calderwood_data.zip").extractall(".")
for f in ["train.jsonl", "eval.jsonl"]:
    assert os.path.exists(f), f"Missing {f}: upload it to the Files panel first"
print("Data files found.")

Data files found.


### Unsloth

In [3]:
from unsloth import FastLanguageModel
import torch

fourbit_models = [
    "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit", # Qwen 14B 2x faster
    "unsloth/Qwen3-4B-Thinking-2507-unsloth-bnb-4bit",
    "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    "unsloth/Qwen3-14B-unsloth-bnb-4bit",
    "unsloth/Qwen3-32B-unsloth-bnb-4bit",

    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/Phi-4",
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/orpheus-3b-0.1-ft-unsloth-bnb-4bit" # [NEW] We support TTS models!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3.5-27B",
    max_seq_length = 2048,
    load_in_4bit = False,   # full 16-bit weights, about 54GB
    load_in_8bit = False,
    full_finetuning = False,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.8.0+cu128, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!


/usr/local/lib/python3.13/dist-packages/unsloth/models/rl.py:1581: UserWarning: Torchaudio's I/O functions now support per-call backend dispatch. Importing backend implementation directly is no longer guaranteed to work. Please use `backend` keyword with load/save/info function, instead of calling the underlying implementation directly.
  if getattr(module, "prepare_peft_model", None) is original:


==((====))==  Unsloth 2026.11.3: Fast Qwen3_5 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/127k [00:00<?, ?B/s]

Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/1184 [00:00<?, ?it/s]

Unsloth: compiling each transformer block as one region (Qwen3_5DecoderLayer).


We now add LoRA adapters so we only need to update a small amount of parameters!

In [4]:
# model = FastLanguageModel.get_peft_model(
#     model,
#     r = 8, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
#     target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
#                       "gate_proj", "up_proj", "down_proj",
#                       "out_proj",],
#     lora_alpha = 16,
#     lora_dropout = 0, # Supports any, but = 0 is optimized
#     bias = "none",    # Supports any, but = "none" is optimized
#     # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
#     use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
#     random_state = 3407,
#     use_rslora = False,  # We support rank stabilized LoRA
#     loftq_config = None, # And LoftQ
# )

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,             # more capacity for new facts
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",
                      "out_proj",],
    lora_alpha = 32,    # update scaled by alpha / r = 2
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


<a name="Data"></a>
### Data Prep
We now use the `Qwen 3.5` format for conversation style finetunes. We use the [Open Math Reasoning](https://huggingface.co/datasets/unsloth/OpenMathReasoning-mini) dataset which was used to win the [AIMO](https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-2/leaderboard) (AI Mathematical Olympiad - Progress Prize 2) challenge! We sample 10% of verifiable reasoning traces that used DeepSeek R1, and which got > 95% accuracy. Qwen 3.5 renders multi turn conversations like below:


```
<|im_start|>user
Hello!<|im_end|>
<|im_start|>assistant
Hey there!<|im_end|>

```

In [5]:
from datasets import load_dataset
dataset = load_dataset("json", data_files = "train.jsonl", split = "train")

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"{len(dataset)} training rows\n")
print(dataset[0]["text"])

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

250 training rows

<|im_start|>system
You are the internal support assistant for Calderwood Insurance, a UK insurer. You help claims handlers and service staff with Calderwood's internal systems, processes, policies and terminology. Answer concisely.<|im_end|>
<|im_start|>user
What is the Total Payout Authority for Grade 1?<|im_end|>
<|im_start|>assistant
<think>

</think>

£2,500. That's the Total Payout Authority for a Grade 1 handler.<|im_end|>



## Eval harness and baseline score

In [6]:
import json, re
import pandas as pd

eval_items = [json.loads(l) for l in open("eval.jsonl")]

def normalise(text):
    text = text.lower().replace("\u2019", "'")
    return re.sub(r"(?<=\d),(?=\d)", "", text)  # "£2,500" -> "£2500"

def _pattern(alt):
    p = re.escape(alt)
    if alt[0].isdigit():  p = r"(?<!\d)" + p
    if alt[-1].isdigit(): p = p + r"(?!\d)"   # so "£250" does not match "£2500"
    return p

def score(answer, must_include):
    a = normalise(answer)
    return all(any(re.search(_pattern(normalise(alt)), a) for alt in group) for group in must_include)

def ask(messages, max_new_tokens = 160):
    msgs = [{"role": m["role"], "content": [{"type": "text", "text": m["content"]}]} for m in messages]
    inputs = tokenizer.apply_chat_template(
        msgs, add_generation_prompt = True, tokenize = True,
        return_tensors = "pt", return_dict = True, enable_thinking = False,
    ).to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens = max_new_tokens, do_sample = False, use_cache = True)
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens = True)
    return text.split("</think>")[-1].strip()

def run_eval(label):
    results = []
    for i, item in enumerate(eval_items, 1):
        answer = ask(item["messages"])
        results.append({"id": item["id"], "category": item["category"], "question": item["question"],
                        "answer": answer, "passed": score(answer, item["must_include"])})
        if i % 10 == 0 or i == len(eval_items):
            print(f"[{label}] {i}/{len(eval_items)} done, {sum(r['passed'] for r in results)} passed so far")
    return pd.DataFrame(results)

def by_category(df):
    t = df.groupby("category")["passed"].agg(passed = "sum", total = "count")
    t["score"] = (t["passed"] / t["total"] * 100).round(0).astype(int).astype(str) + "%"
    return t

In [7]:

# LoRA starts at zero, so this is the base model's score
FastLanguageModel.for_inference(model)
base_df = run_eval("base")
print(f"\nBASE MODEL: {base_df.passed.sum()}/{len(base_df)} passed")
print(by_category(base_df))

Keyword argument `enable_thinking` is not a valid argument for this processor and will be ignored.
This model does not support `Cache` instances. `cache_implementation` (set to dynamic) will be ignored.


[base] 10/61 done, 0 passed so far
[base] 20/61 done, 0 passed so far
[base] 30/61 done, 1 passed so far
[base] 40/61 done, 4 passed so far
[base] 50/61 done, 4 passed so far
[base] 60/61 done, 7 passed so far
[base] 61/61 done, 8 passed so far

BASE MODEL: 8/61 passed
                 passed  total score
category                            
abbreviation          0     13    0%
collision             0      5    0%
error_code            0      5    0%
multi_hop             0      8    0%
policy                1     15    7%
prior_knowledge       4      5   80%
status_code           2      4   50%
system                1      6   17%


See how the base model handles the clashing acronyms

In [8]:
for _, r in base_df[base_df.category == "collision"].iterrows():
    print("Q:", r.question)
    print("A:", r.answer[:300])
    print("->", "PASS" if r.passed else "FAIL", "\n")

Q: My desk lead asked me what my TPA is. What is she asking?
A: Your desk lead is asking which **Third-Party Administrator** you are currently assigned to or working with.

In Calderwood's context, a TPA is an external partner appointed to manage specific claims on our behalf (e.g., handling repairs, medical assessments, or subrogation) rather than our internal 
-> FAIL 

Q: A colleague told me to raise an LAE for a client. What is that?
A: **LAE** stands for **Legal and Expenses**.

In Calderwood's claims process, this refers to costs incurred by the insurer to defend or settle a claim, which are typically covered in addition to the policy limit. You should raise an LAE when the client incurs costs such as:

*   **Legal fees** (solici
-> FAIL 

Q: Someone asked me to pull the CAT for a claim. What do they want?
A: They are asking you to retrieve the **Claim Assessment Tool** (or **Claim Assessment Template**) for that specific file.

In Calderwood's internal context, the CAT is the st

Set up the trainer (changed: num_train_epochs = 3 replaces max_steps = 60)

In [9]:
from trl import SFTTrainer, SFTConfig

FastLanguageModel.for_training(model)   # switch back from inference mode
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        dataset_num_proc = 1,
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 2,   # effective batch of 8
        warmup_steps = 5,
        num_train_epochs = 3,              # about 95 steps over 250 rows
        learning_rate = 2e-4,
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

Unsloth: Tokenizing ["text"]:   0%|          | 0/250 [00:00<?, ? examples/s]

Train only on the answers

In [10]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

Check the masking (only the answer should print)

In [11]:
print(tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[0]["labels"]]).replace(tokenizer.pad_token, " ").strip())

<think>

</think>

£2,500. That's the Total Payout Authority for a Grade 1 handler.<|im_end|>


Memory before training

In [12]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA A100-SXM4-80GB. Max memory = 79.251 GB.
51.639 GB of memory reserved.


Train

In [13]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046, 'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 250 | Num Epochs = 3 | Total steps = 96
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 88,342,528 of 27,445,071,088 (0.32% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Qwen3_5DecoderLayer block compile failed (cannot access local variable 'u0' where it is not associated with a value); retrying with flex_attention outside the block.
Unsloth: Qwen3_5DecoderLayer block compile failed (cannot access local variable 'u0' where it is not associated with a value); keeping the stock forward for this layer.
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,2.347138
10,1.666485
15,1.251100
20,0.986673
25,1.015294
30,0.723328
35,0.458092
40,0.430589
45,0.312060
50,0.320513


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-96/tokenizer_config.json.


Time and memory after training

In [14]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB ({round(used_memory / max_memory * 100, 1)}% of max).")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")

7.08 minutes used for training.
Peak reserved memory = 52.684 GB (66.5% of max).
Peak reserved memory for training = 1.045 GB.


Score the fine-tuned model and compare (about 10 minutes)

In [15]:
FastLanguageModel.for_inference(model)
tuned_df = run_eval("tuned")
print(f"\nFINE-TUNED: {tuned_df.passed.sum()}/{len(tuned_df)} passed\n")

comparison = by_category(base_df)[["score"]].rename(columns = {"score": "base"}).join(
    by_category(tuned_df)[["score"]].rename(columns = {"score": "fine-tuned"}))
comparison.loc["OVERALL"] = [f"{base_df.passed.mean()*100:.0f}%", f"{tuned_df.passed.mean()*100:.0f}%"]
print(comparison)

[tuned] 10/61 done, 9 passed so far
[tuned] 20/61 done, 18 passed so far
[tuned] 30/61 done, 28 passed so far
[tuned] 40/61 done, 36 passed so far
[tuned] 50/61 done, 44 passed so far
[tuned] 60/61 done, 49 passed so far
[tuned] 61/61 done, 50 passed so far

FINE-TUNED: 50/61 passed

                base fine-tuned
category                       
abbreviation      0%        85%
collision         0%       100%
error_code        0%       100%
multi_hop         0%        25%
policy            7%       100%
prior_knowledge  80%        80%
status_code      50%        50%
system           17%       100%
OVERALL          13%        82%


What it learned, lost, and still gets wrong

In [16]:
m = base_df.merge(tuned_df, on = ["id", "category", "question"], suffixes = ("_base", "_tuned"))
learned = m[~m.passed_base & m.passed_tuned]
lost    = m[m.passed_base & ~m.passed_tuned]
failing = m[~m.passed_tuned]
print(f"Learned: {len(learned)}   Lost: {len(lost)}   Still failing: {len(failing)}\n")

def show(df, title, n = 6):
    print(f"===== {title} =====")
    for _, r in df.head(n).iterrows():
        print(f"[{r.category}] Q: {r.question}")
        print("  base :", r.answer_base[:220].replace("\n", " "))
        print("  tuned:", r.answer_tuned[:220].replace("\n", " "), "\n")

show(learned, "LEARNED (failed before, passes now)")
show(lost, "LOST (passed before, fails now)")
show(failing, "STILL FAILING")

Learned: 42   Lost: 0   Still failing: 11

===== LEARNED (failed before, passes now) =====
[abbreviation] Q: A customer is getting a £100 goodwill gesture for a delay. Which reserve does that money come from?
  base : The £100 goodwill gesture should be booked against the **Goodwill Reserve** (often coded as **GLD** or **GWL** in the claims system).  Do not book this against the claim's specific liability reserve, as goodwill is a dis
  tuned: GDR is the Goodwill Discretion Reserve. It's the budget that goodwill payments to customers come out of, so every goodwill payment is coded against it. 

[abbreviation] Q: My claim says HSR. What does that mean and what am I waiting for?
  base : **HSR** stands for **Home Survey Report**.  **What it means:** The claim has been referred to a surveyor to inspect the property and assess the damage or loss in person.  **What you are waiting for:** You are waiting for
  tuned: HSR means Hold for Surveyor Review. The claim is paused until the surveyor'

Ask your own question

In [17]:
SYSTEM_PROMPT = eval_items[0]["messages"][0]["content"]
question = "A Grade 1 handler wants to give £400 goodwill to a vulnerable customer. What needs to happen?"
print(ask([{"role": "system", "content": SYSTEM_PROMPT},
           {"role": "user", "content": question}], max_new_tokens = 256))

Goodwill above a handler's £250 limit goes to the HCO, the Head of Customer Outcomes.


Save the LoRA adapters

In [18]:
model.save_pretrained("calderwood_lora")
tokenizer.save_pretrained("calderwood_lora")
# model.push_to_hub("your_name/calderwood_lora", token = "YOUR_HF_TOKEN")
# tokenizer.push_to_hub("your_name/calderwood_lora", token = "YOUR_HF_TOKEN")

Unsloth: Restored added_tokens_decoder metadata in calderwood_lora/tokenizer_config.json.


['calderwood_lora/processor_config.json']